# Importing the dependencies

In [12]:
import os
import pandas as pd
import numpy as np
import librosa
from tqdm import tqdm

In [25]:
PROJECT_ROOT = "/Users/dwibon/Desktop/IKS-Internship"
os.chdir(PROJECT_ROOT)
 
SEGMENT_METADATA_PATH = "dataset_segments/segment_metadata.csv"
OUTPUT_PATH = "dataset_segments/segment_features.csv"
TARGET_SR = 22050
N_MFCC = 13

In [14]:
def extract_time_domain(y):
    rms = librosa.feature.rms(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    return {
        "time_rms_mean": float(rms.mean()), "time_rms_std": float(rms.std()),
        "time_zcr_mean": float(zcr.mean()), "time_zcr_std": float(zcr.std()),
    }

In [15]:
def extract_spectral(y, sr):
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    return {
        "spectral_centroid_mean": float(centroid.mean()), "spectral_centroid_std": float(centroid.std()),
        "spectral_rolloff_mean": float(rolloff.mean()), "spectral_rolloff_std": float(rolloff.std()),
        "spectral_bandwidth_mean": float(bandwidth.mean()), "spectral_bandwidth_std": float(bandwidth.std()),
    }

In [16]:
def extract_mfcc(y, sr, n_mfcc=N_MFCC):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    feats = {}
    for i in range(n_mfcc):
        feats[f"mfcc_{i+1}_mean"] = float(mfcc[i].mean())
        feats[f"mfcc_{i+1}_std"] = float(mfcc[i].std())
    return feats

In [17]:
def extract_chroma(y, sr):
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=12)
    feats = {}
    for i in range(12):
        feats[f"chroma_{i+1}_mean"] = float(chroma[i].mean())
        feats[f"chroma_{i+1}_std"] = float(chroma[i].std())
    return feats

In [18]:
def extract_tempo(y, sr):
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    tempo_val = float(np.ravel(tempo)[0]) if np.size(tempo) > 0 else 0.0
    return {"tempo_bpm": tempo_val}

In [26]:
def main():
    meta = pd.read_csv(SEGMENT_METADATA_PATH)
    rows = []
    errors = []
 
    for _, row in tqdm(meta.iterrows(), total=len(meta), desc="Extracting features"):
        path = row["segment_filepath"]
        try:
            y, sr = librosa.load(path, sr=TARGET_SR, mono=True)
        except Exception as e:
            errors.append((row["segment_id"], str(e)))
            continue
 
        feats = {}
        feats.update(extract_time_domain(y))
        feats.update(extract_spectral(y, sr))
        feats.update(extract_mfcc(y, sr))
        feats.update(extract_chroma(y, sr))
        feats.update(extract_tempo(y, sr))
 
        out_row = row.to_dict()
        out_row.update(feats)
        rows.append(out_row)
 
    feat_df = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(OUTPUT_PATH) or ".", exist_ok=True)
    feat_df.to_csv(OUTPUT_PATH, index=False)
 
    print(f"\n{'='*50}")
    print(f"Done. {len(feat_df)}/{len(meta)} segments processed.")
    print(f"Saved to: {OUTPUT_PATH}")
    if errors:
        print(f"\n{len(errors)} segment(s) failed:")
        for sid, err in errors:
            print(f"  {sid}: {err}")
 
 
if __name__ == "__main__":
    main()
 

Extracting features: 100%|██████████████████| 3088/3088 [10:17<00:00,  5.00it/s]



Done. 3088/3088 segments processed.
Saved to: dataset_segments/segment_features.csv
